In [ ]:
import sys
import os

print("Python:", sys.version)
print("Current folder:", os.getcwd())

Python: 3.13.11 (tags/v3.13.11:6278944, Dec  5 2025, 16:26:58) [MSC v.1944 64 bit (AMD64)]
Current folder: C:\Users\Dell\OneDrive\Desktop\Design thinking LLM\air_quality_project


In [ ]:
import os
import pandas as pd
import numpy as np

print("Files inside data folder:")
print(os.listdir("data"))

df = pd.read_csv(
    "data/AirQualityUCI.csv",
    sep=";",
    decimal=","
)

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Files inside data folder:
['AirQualityUCI.csv', 'AirQualityUCI.xlsx']

Dataset shape: (9471, 17)

Columns:
['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'Unnamed: 15', 'Unnamed: 16']


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


In [ ]:
import numpy as np
import pandas as pd

# Remove empty unnamed columns
df = df.drop(
    columns=[
        col for col in df.columns
        if col.startswith("Unnamed")
    ]
)

# Remove completely empty rows
df = df.dropna(how="all").copy()

# UCI Air Quality dataset uses -200 as "missing"
df = df.replace(-200, np.nan)

print("Cleaned shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

df.head()

Cleaned shape: (9357, 15)

Columns:
['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']

Missing values:
Date                0
Time                0
CO(GT)           1683
PT08.S1(CO)       366
NMHC(GT)         8443
C6H6(GT)          366
PT08.S2(NMHC)     366
NOx(GT)          1639
PT08.S3(NOx)      366
NO2(GT)          1642
PT08.S4(NO2)      366
PT08.S5(O3)       366
T                 366
RH                366
AH                366
dtype: int64


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [ ]:
features = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
    "T",
    "RH",
    "AH"
]

target = "CO(GT)"

llm_df = df[
    ["Date", "Time"] + features + [target]
].copy()

# Remove rows where required values are missing
llm_df = llm_df.dropna().reset_index(drop=True)

print("Original cleaned rows:", len(df))
print("Usable LLM training rows:", len(llm_df))

llm_df.head()

Original cleaned rows: 9357
Usable LLM training rows: 7344


,Date,Time,PT08.S1(CO),PT08.S2(NMHC),PT08.S3(NOx),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,CO(GT)
0,10/03/2004,18.00.00,1360.0,1046.0,1056.0,1692.0,1268.0,13.6,48.9,0.7578,2.6
1,10/03/2004,19.00.00,1292.0,955.0,1174.0,1559.0,972.0,13.3,47.7,0.7255,2.0
2,10/03/2004,20.00.00,1402.0,939.0,1140.0,1555.0,1074.0,11.9,54.0,0.7502,2.2
3,10/03/2004,21.00.00,1376.0,948.0,1092.0,1584.0,1203.0,11.0,60.0,0.7867,2.2
4,10/03/2004,22.00.00,1272.0,836.0,1205.0,1490.0,1110.0,11.2,59.6,0.7888,1.6


In [ ]:
def create_training_text(row):

    return f"""### Instruction:
Predict the carbon monoxide concentration CO(GT) from the given air-quality sensor readings.

### Input:
Date: {row['Date']}
Time: {row['Time']}
PT08.S1(CO): {row['PT08.S1(CO)']}
PT08.S2(NMHC): {row['PT08.S2(NMHC)']}
PT08.S3(NOx): {row['PT08.S3(NOx)']}
PT08.S4(NO2): {row['PT08.S4(NO2)']}
PT08.S5(O3): {row['PT08.S5(O3)']}
Temperature: {row['T']} C
Relative Humidity: {row['RH']} %
Absolute Humidity: {row['AH']}

### Response:
CO(GT): {row['CO(GT)']}"""


llm_df["text"] = llm_df.apply(
    create_training_text,
    axis=1
)

print(llm_df["text"].iloc[0])

### Instruction:
Predict the carbon monoxide concentration CO(GT) from the given air-quality sensor readings.

### Input:
Date: 10/03/2004
Time: 18.00.00
PT08.S1(CO): 1360.0
PT08.S2(NMHC): 1046.0
PT08.S3(NOx): 1056.0
PT08.S4(NO2): 1692.0
PT08.S5(O3): 1268.0
Temperature: 13.6 C
Relative Humidity: 48.9 %
Absolute Humidity: 0.7578

### Response:
CO(GT): 2.6


In [ ]:
import json

output_path = "data/air_quality_llm_train.jsonl"

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    for text in llm_df["text"]:

        json.dump(
            {"text": text},
            f,
            ensure_ascii=False
        )

        f.write("\n")


print("Saved:", output_path)
print("Number of examples:", len(llm_df))

Saved: data/air_quality_llm_train.jsonl
Number of examples: 7344


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    vram = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(
        f"VRAM: {vram:.2f} GB"
    )

PyTorch version: 2.14.0+cpu
CUDA available: False
